In [63]:
import pandas as pd
from sqlalchemy import create_engine
from config import *
from mapping import COLUMN_MAPPING

In [75]:
df = pd.read_csv("../../Dataset/Processed/featured_sales.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Order Month,Order Month Name,Order Quarter,Order Day,Order Weekday,Shipping Days,Profit Margin,Has Discount,High Profit,Order Size
0,1,INDMKB,2020-11-08,2020-11-11,sas,as,asa,asa,asas,asas,...,11,November,4,8,Sunday,3,16.00,No,High,Large
1,2,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,asasa,Consumer,United States,Henderson,...,11,November,4,8,Sunday,3,30.00,No,High,Large
2,3,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,6,June,2,12,Friday,4,47.00,No,Low,Small
3,4,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,10,October,4,11,Friday,7,-40.00,Yes,Low,Large
4,5,US-2019-108966,2019-10-11,2019-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,10,October,4,11,Friday,7,11.25,Yes,Low,Small


In [78]:
print(df[["Has Discount", "High Profit"]].head(10))
print(df["Has Discount"].unique())

print(df["High Profit"].unique())

  Has Discount High Profit
0           No        High
1           No        High
2           No         Low
3          Yes         Low
4          Yes         Low
5           No        High
6           No         Low
7          Yes        High
8          Yes         Low
9           No        High
['No' 'Yes']
['High' 'Low']


In [79]:
df.rename(columns=COLUMN_MAPPING, inplace=True)

print(df.columns.tolist())

['RowID', 'OrderID', 'OrderDate', 'ShipDate', 'ShipMode', 'CustomerID', 'CustomerName', 'Segment', 'Country', 'City', 'State', 'PostalCode', 'Region', 'ProductID', 'Category', 'SubCategory', 'ProductName', 'Sales', 'Quantity', 'Discount', 'Profit', 'OrderYear', 'OrderMonth', 'OrderMonthName', 'OrderQuarter', 'OrderDay', 'OrderWeekday', 'ShippingDays', 'ProfitMargin', 'HasDiscount', 'HighProfit', 'OrderSize']


In [ ]:
# from sqlalchemy import create_engine

# engine = create_engine(
#     "mssql+pyodbc://MOHAMED-AHMED/ECommerceAnalytics?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
# )

# with engine.connect() as conn:
#     print("Connection Successful!")

Connection Successful!


In [80]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["ShipDate"] = pd.to_datetime(df["ShipDate"])

In [81]:
df["HasDiscount"] = (
    df["HasDiscount"]
    .astype(str)
    .str.strip()
    .map({
        "Yes": True,
        "No": False
    })
)

df["HighProfit"] = (
    df["HighProfit"]
    .astype(str)
    .str.strip()
    .map({
        "High": True,
        "Low": False
    })
)


In [82]:
print(df.dtypes)

print(df.isnull().sum())

RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDays               int64
ProfitMargin             float64
HasDiscount                 bool
HighProfit

In [83]:
connection_string = (
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver={DRIVER.replace(' ', '+')}"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

with engine.connect() as conn:
    print("Connection Successful!")

Connection Successful!


In [84]:
print(df["HasDiscount"].unique())
print(df["HighProfit"].unique())

print(df.dtypes)

[False  True]
[ True False]
RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDays               int64
ProfitMargin             float64
HasDiscount    

In [85]:
print("=" * 50)
print("Data Validation Summary")
print("=" * 50)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Data Validation Summary
Rows: 9994
Columns: 32

Missing Values:
0

Duplicate Rows:
0

Data Types:
RowID                      int64
OrderID                   object
OrderDate         datetime64[ns]
ShipDate          datetime64[ns]
ShipMode                  object
CustomerID                object
CustomerName              object
Segment                   object
Country                   object
City                      object
State                     object
PostalCode                object
Region                    object
ProductID                 object
Category                  object
SubCategory               object
ProductName               object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
OrderYear                  int64
OrderMonth                 int64
OrderMonthName            object
OrderQuarter               int64
OrderDay                   int64
OrderWeekday              object
ShippingDay

In [86]:
print(df.shape)

(9994, 32)


In [88]:
df.to_sql(
    name="Sales",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=50  # قللناه عشان نضمن إنه ما يتخطاش الـ 2100 (50 * 32 = 1600)
)

print("Data Loaded Successfully!")

Data Loaded Successfully!
